# Finding Similar issues by semantics

We will use **sentence-transformers**. Install it in your virtual environment using `pip install sentence-transformers`.

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## Loading github issues for transfomer library

In [2]:
pd.set_option('display.max_colwidth', None)

# Load dataset
df = pd.read_csv("./gitissues.csv")

df[['title', 'comments']].sample(10)

,title,comments
969,Filter on dataset too much slowww,"When I use the filter on the arrow table directly, it works like butter. But I can't find a way to update the table in `Dataset` object.\r\n\r\n```\r\nds_table = dataset.data.filter(mask=dataset['flag'])\r\n```"
2118,Meta-datasets (GLUE/XTREME/...) – Special care to attributions and citations,Looks good but why would there be a difference between the ref in the source and the one to be printed?
1907,ERROR:root:mwparserfromhell,"Thanks a lot @Shiro-LK !\r\n\r\nI was able to reproduce the issue. It comes from [this table on wikipedia](https://es.wikipedia.org/wiki/Campeonato_Mundial_de_futsal_de_la_AMF_2015#Clasificados) that can't be parsed.\r\n\r\nThe file in which the problem occurs comes from the wikipedia dumps, and it can be downloaded [here](https://dumps.wikimedia.org/eswiki/20200501/eswiki-20200501-pages-articles-multistream6.xml-p6424816p7924815.bz2)\r\n\r\nParsing the file this way raises the parsing issue:\r\n\r\n```python\r\nimport mwparserfromhell as parser\r\nfrom tqdm.auto import tqdm\r\nimport bz2\r\nimport six\r\nimport logging\r\nimport codecs\r\nimport xml.etree.cElementTree as etree\r\n\r\nfilepath = ""path/to/eswiki-20200501-pages-articles-multistream6.xml-p6424816p7924815.bz2""\r\n\r\ndef _extract_content(filepath):\r\n """"""Extracts article content from a single WikiMedia XML file.""""""\r\n logging.info(""generating examples from = %s"", filepath)\r\n with open(filepath, ""rb"") as f:\r\n f = bz2.BZ2File(filename=f)\r\n if six.PY3:\r\n # Workaround due to:\r\n # https://github.com/tensorflow/tensorflow/issues/33563\r\n utf_f = codecs.getreader(""utf-8"")(f)\r\n else:\r\n utf_f = f\r\n # To clear root, to free-up more memory than just `elem.clear()`.\r\n context = etree.iterparse(utf_f, events=(""end"",))\r\n context = iter(context)\r\n unused_event, root = next(context)\r\n for unused_event, elem in tqdm(context, total=949087):\r\n if not elem.tag.endswith(""page""):\r\n continue\r\n namespace = elem.tag[:-4]\r\n title = elem.find(""./{0}title"".format(namespace)).text\r\n ns = elem.find(""./{0}ns"".format(namespace)).text\r\n id_ = elem.find(""./{0}id"".format(namespace)).text\r\n # Filter pages that are not in the ""main"" namespace.\r\n if ns != ""0"":\r\n root.clear()\r\n continue\r\n raw_content = elem.find(""./{0}revision/{0}text"".format(namespace)).text\r\n root.clear()\r\n\r\n if ""Campeonato Mundial de futsal de la AMF 2015"" in title:\r\n yield (id_, title, raw_content)\r\n\r\nfor id_, title, raw_content in _extract_content(filepath):\r\n wikicode = parser.parse(raw_content)\r\n```\r\n\r\nThe copied the raw content that can't be parsed [here](https://pastebin.com/raw/ZbmevLyH).\r\n\r\nThe minimal code to reproduce is:\r\n```python\r\nimport mwparserfromhell as parser\r\nimport requests\r\n\r\nraw_content = requests.get(""https://pastebin.com/raw/ZbmevLyH"").content.decode(""utf-8"")\r\nwikicode = parser.parse(raw_content)\r\n\r\n```\r\n\r\nI will create an issue on mwparserfromhell's repo to see if we can fix that\r\n"
1789,UnicodeDecodeError while loading PAN-X task of XTREME dataset,"I've created a simple function that seems to do the trick:\r\n\r\n```python\r\ndef apply_encoding_on_file_open(filepath: str):\r\n """"""Apply UTF-8 encoding for all instances where a non-binary file is opened.""""""\r\n \r\n with open(filepath, 'r', encoding='utf-8') as input_file:\r\n regexp = re.compile(r""""""\r\n (?!.*\b(?:encoding|rb|wb|wb+|ab|ab+)\b)\r\n (open)\r\n \((.*)\)\r\n """""")\r\n input_text = input_file.read()\r\n match = regexp.search(input_text)\r\n \r\n if match:\r\n print('Found match!', match.group())\r\n # append utf-8 encoding to matching groups in-place\r\n output = regexp.sub(lambda m: m.group()[:-1]+', encoding=""utf-8"")', input_text)\r\n with open(filepath, 'w', encoding='utf-8') as output_file:\r\n output_file.write(output)\r\n else:\r\n print(""No match found!"")\r\n```\r\n\r\nThe regexp does a negative lookahead to avoid matc

## Embedding the issues

In [3]:
def embed_issues(df, column_to_embed='title', model_name='all-MiniLM-L6-v2'):
    """
    Embeds the selected text column using SentenceTransformer.

    Parameters:
    - df: DataFrame containing the GitHub issues.
    - column_to_embed: Column name to embed ('title', 'body', etc.).
    - model_name: Pre-trained model name from sentence-transformers.

    Returns:
    - embeddings: numpy array of sentence embeddings.
    - model: the loaded SentenceTransformer model.
    """
    model = SentenceTransformer(model_name)
    texts = df[column_to_embed].fillna('').tolist()
    embeddings = model.encode(texts, convert_to_numpy=True)
    return embeddings, model

In [4]:
# Configuration
column = 'title'

# Step 1: Embed dataset
issue_embeddings, embed_model = embed_issues(df, column_to_embed=column)

In [6]:
issue_embeddings.shape

(2175, 384)

## Finding similar issues

In [9]:
def search_similar_issues(df, embeddings, model, problem_description, k=5, column_to_embed='title'):
    """
    Finds top-k similar issues from the dataset.

    Parameters:
    - df: DataFrame with GitHub issues.
    - embeddings: Precomputed issue embeddings.
    - model: SentenceTransformer model used for embedding.
    - problem_description: Query string to compare against dataset.
    - k: Number of top similar issues to return.
    - column_to_embed: For display context, can be title or body.

    Returns:
    - DataFrame with top-k similar issues and similarity scores.
    """
    query_embedding = model.encode([problem_description], convert_to_numpy=True)
    sim_scores = cosine_similarity(query_embedding, embeddings)[0]
    top_k_idx = np.argsort(sim_scores)[-k:][::-1]

    result_df = df.iloc[top_k_idx].copy()
    result_df['similarity'] = sim_scores[top_k_idx]
    return result_df[['title', 'similarity', 'comments']]

## Seach similar problems

In [10]:
k = 10

query = "caching configuration not working"

# Step 2: Search similar issues
similar = search_similar_issues(df,
                                issue_embeddings,
                                embed_model,
                                query,
                                k=k,
                                column_to_embed=column)

similar

,title,similarity,comments
1424,Possible caching bug,0.672568,Thanks for reporting. That's a bug indeed.\r\nApparently only the `data_files` parameter is taken into account right now in `DatasetBuilder._create_builder_config` but it should also be the case for `config_kwargs` (or at least the instantiated `builder_config`)
1757,Caching doesn't work for map (non-deterministic),0.620773,"Thanks for reporting !\r\n\r\nTo store the cache file, we compute a hash of the function given in `.map`, using our own hashing function.\r\nThe hash doesn't seem to stay the same over sessions for the tokenizer.\r\nApparently this is because of the regex at `tokenizer.pat` is not well supported by our hashing function.\r\n\r\nI'm working on a fix"
150,Missing cache file,0.613573,"This could be solved by going to the glue/ directory and delete sst2 directory, then load the dataset again will help you redownload the dataset."
151,Missing cache file,0.613573,"Hi ! Not sure why this file was missing, but yes the way to fix this is to delete the sst2 directory and to reload the dataset"
349,Cached dataset not loaded,0.578329,"> does it consider just the name or the actual code of the function\r\n\r\nIt looks at the name and the actual code and all variables such as recursively. It uses `dill` to do so, which is based on `pickle`.\r\nBasically the hash is computed using the pickle bytes of your function (computed using `dill` to support most python objects).\r\n\r\n> does it consider variables that are not passed explicitly as parameters to the functions (such as the processor here)\r\n\r\nYes it does thanks to recursive pickling."
346,Cached dataset not loaded,0.578329,It actually seems to happen all the time in above configuration:\r\n* the function `filter_by_duration` correctly loads cached processed dataset\r\n* the function `prepare_dataset` is always reexecuted\r\n\r\nI end up solving the issue by saving to disk my dataset at the end but I'm still wondering if it's a bug or limitation here.
348,Cached dataset not loaded,0.578329,"> If `prepare_dataset` is always executed, maybe this is because your `processor` has a different hash each time you want to execute it.\r\n\r\nYes I think that was the issue.\r\n\r\nFor the hash of the function:\r\n* does it consider just the name or the actual code of the function\r\n* does it consider variables that are not passed explicitly as parameters to the functions (such as the processor here)"
347,Cached dataset not loaded,0.578329,"Hi ! The hash used for caching `map` results is the fingerprint of the resulting dataset. It is computed using three things:\r\n- the old fingerprint of the dataset\r\n- the hash of the function\r\n- the hash of the other parameters passed to `map`\r\n\r\nYou can compute the hash of your function (or any python object) with\r\n```python\r\nfrom datasets.fingerprint import Hasher\r\n\r\nmy_func = lambda x: x + 1\r\nprint(Hasher.hash(my_func))\r\n```\r\n\r\nIf `prepare_dataset` is always executed, maybe this is because your `processor` has a different hash each time you want to execute it."
1507,Caching processed dataset at wrong folder,0.567680,Actually I did more tests it doesn't >.<\r\nI'll let you know if I find a way to fix that
1509,Caching processed dataset at wrong folder,0.567680,The code you wrote should write a 1GB file in the Google Drive folder. Doesn't it?
